In [1]:
pip install pandas pyarrow pyyaml pandera

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import os
from pathlib import Path

import pandas as pd
import yaml
import pandera.pandas as pa
from pandera.pandas import Check, Column, DataFrameSchema

BASE = Path(os.getcwd())
CFG = BASE / "config" / "pipeline.yaml"

print("Current working folder:", BASE)
print("Config file path:", CFG)

Current working folder: c:\Users\Lenovo\Downloads\finmark_pipeline
Config file path: c:\Users\Lenovo\Downloads\finmark_pipeline\config\pipeline.yaml


In [3]:
def load_cfg():
    with open(CFG, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
    return df


def clean_strings(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == "object":
            df[c] = df[c].astype(str).str.strip()
            df.loc[df[c].isin(["", "nan", "None", "null"]), c] = pd.NA
    return df


def coerce_datetime(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in df.columns:
        lc = c.lower()
        if any(k in lc for k in ["time", "date", "timestamp"]):
            try:
                converted = pd.to_datetime(df[c], errors="coerce")
                if converted.notna().sum() > 0:
                    df[c] = converted
            except Exception:
                pass
    return df


def fill_nulls(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].fillna(0)
        elif pd.api.types.is_datetime64_any_dtype(df[c]):
            continue
        else:
            df[c] = df[c].fillna("unknown")
    return df

In [4]:
def build_schema(df: pd.DataFrame) -> DataFrameSchema:
    schema_columns = {}

    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            schema_columns[col] = Column(
                pa.DateTime,
                nullable=True,
                required=True
            )

        elif pd.api.types.is_numeric_dtype(df[col]):
            schema_columns[col] = Column(
                float if "float" in str(df[col].dtype) else int,
                nullable=False,
                required=True
            )

        else:
            schema_columns[col] = Column(
                str,
                nullable=False,
                required=True,
                checks=Check.str_length(min_value=1)
            )

    schema = DataFrameSchema(
        schema_columns,
        strict=False,
        coerce=True
    )

    return schema

In [5]:
cfg = load_cfg()

raw = BASE / "data" / "raw" / "event_logs.csv"
out_csv = BASE / "data" / "processed" / "cleaned_event_logs.csv"
out_parquet = BASE / "data" / "processed" / "cleaned_event_logs.parquet"
out_json = BASE / "data" / "processed" / "pipeline_summary.json"

print("Raw file:", raw)
print("Output CSV:", out_csv)
print("Output Parquet:", out_parquet)
print("Output JSON:", out_json)

Raw file: c:\Users\Lenovo\Downloads\finmark_pipeline\data\raw\event_logs.csv
Output CSV: c:\Users\Lenovo\Downloads\finmark_pipeline\data\processed\cleaned_event_logs.csv
Output Parquet: c:\Users\Lenovo\Downloads\finmark_pipeline\data\processed\cleaned_event_logs.parquet
Output JSON: c:\Users\Lenovo\Downloads\finmark_pipeline\data\processed\pipeline_summary.json


In [6]:
df = pd.read_csv(raw)

rows_in = len(df)
cols_in = len(df.columns)
nulls_before = int(df.isna().sum().sum())

df = normalize_columns(df)
df = clean_strings(df)
df = coerce_datetime(df)

dup_count = int(df.duplicated().sum())
df = df.drop_duplicates()

df = fill_nulls(df)

schema = build_schema(df)
validated_df = schema.validate(df)

rows_out = len(validated_df)
cols_out = len(validated_df.columns)
nulls_after = int(validated_df.isna().sum().sum())

out_csv.parent.mkdir(parents=True, exist_ok=True)

validated_df.to_csv(out_csv, index=False)
validated_df.to_parquet(out_parquet, index=False)

summary = {
    "project": cfg["project"]["name"],
    "rows_in": rows_in,
    "rows_out": rows_out,
    "columns_in": cols_in,
    "columns_out": cols_out,
    "nulls_before": nulls_before,
    "nulls_after": nulls_after,
    "duplicates_removed": dup_count,
    "sample_columns": list(validated_df.columns[:12]),
    "schema_validation": "passed"
}

with open(out_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4, default=str)

print("Pipeline completed successfully.")
print("Schema validation passed.")
print(f"Cleaned CSV saved to: {out_csv}")
print(f"Cleaned Parquet saved to: {out_parquet}")
print(f"Summary JSON saved to: {out_json}")

Pipeline completed successfully.
Schema validation passed.
Cleaned CSV saved to: c:\Users\Lenovo\Downloads\finmark_pipeline\data\processed\cleaned_event_logs.csv
Cleaned Parquet saved to: c:\Users\Lenovo\Downloads\finmark_pipeline\data\processed\cleaned_event_logs.parquet
Summary JSON saved to: c:\Users\Lenovo\Downloads\finmark_pipeline\data\processed\pipeline_summary.json


In [7]:
validated_df.head()

,user_id,event_type,event_time,product_id,amount,col_6,col_7,col_8,col_9,col_10,col_11,col_12,col_13,col_14,col_15,col_16,col_17,col_18,col_19,col_20,col_21,col_22,col_23,col_24,col_25,col_26,col_27,col_28,col_29,col_30,col_31,col_32,col_33,col_34,col_35,col_36,col_37,col_38,col_39,col_40,col_41,col_42,col_43,col_44,col_45,col_46,col_47,col_48,col_49,col_50
0,U0099,checkout,2023-06-03 04:13:00,P010,0.00,C,13.05,0.0,B,0.81,155.0,B,0.00,0.0,unknown,0.00,0.0,unknown,35.34,629.0,unknown,86.57,341.0,C,0.00,473.0,B,0.00,0.0,A,0.00,39.0,A,66.34,130.0,B,0.00,0.0,B,79.32,0.0,B,39.66,138.0,C,0.00,902.0,A,0.00,0.0
1,U0240,wishlist_add,2023-06-03 05:08:00,P020,2900.63,unknown,0.00,0.0,C,0.00,0.0,C,30.72,440.0,B,58.21,165.0,C,98.23,249.0,C,0.00,0.0,B,0.00,586.0,A,0.00,875.0,C,25.86,509.0,A,48.62,0.0,unknown,28.49,693.0,unknown,0.00,714.0,unknown,39.97,507.0,B,0.00,632.0,unknown,38.45,890.0
2,U0374,profile_update,2023-06-05 06:22:00,P028,0.00,A,0.00,0.0,B,60.06,2.0,C,0.00,302.0,unknown,66.66,114.0,A,28.32,247.0,A,0.00,0.0,unknown,51.54,190.0,unknown,16.59,871.0,A,28.23,0.0,B,82.94,0.0,C,0.00,0.0,B,0.00,0.0,unknown,0.00,293.0,unknown,0.00,394.0,unknown,0.00,490.0
3,U0122,page_view,2023-06-06 03:45:00,P001,0.00,C,0.00,747.0,B,0.00,758.0,A,78.44,849.0,unknown,87.51,0.0,A,33.97,217.0,C,86.57,623.0,C,46.57,0.0,unknown,36.31,0.0,C,52.44,921.0,unknown,83.77,789.0,unknown,0.00,936.0,unknown,34.84,365.0,C,67.84,705.0,A,96.06,110.0,unknown,0.00,0.0
4,U0211,wishlist_add,2023-06-03 12:38:00,P015,1728.27,A,40.19,515.0,A,0.00,63.0,B,0.00,851.0,C,12.15,0.0,C,19.60,0.0,A,77.96,536.0,unknown,0.00,0.0,C,30.45,0.0,C,0.00,715.0,C,8.26,152.0,unknown,86.18,173.0,unknown,0.00,0.0,C,0.00,876.0,unknown,0.00,0.0,B,0.00,0.0


In [8]:
summary

{'project': 'finmark-data-pipeline',
 'rows_in': 2000,
 'rows_out': 2000,
 'columns_in': 50,
 'columns_out': 50,
 'nulls_before': 26671,
 'nulls_after': 0,
 'duplicates_removed': 0,
 'sample_columns': ['user_id',
  'event_type',
  'event_time',
  'product_id',
  'amount',
  'col_6',
  'col_7',
  'col_8',
  'col_9',
  'col_10',
  'col_11',
  'col_12'],
 'schema_validation': 'passed'}